# Charts: the waiting list

**Steven**, Data Visualisation 4, The Foreigners Team

Question (Q3, context): how has the number of people on the outpatient waiting list, and how long they wait, changed since 2019?

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

PREPARED = Path("data/prepared")
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

ORANGE = "#eb6834"  # waiting
BLUE = "#2a78d6"  # cost
GREY = "#898781"  # EU average and background

waiting = pd.read_csv(PREPARED / "waiting_list_national_v1_2026-11-11.csv", parse_dates=["snapshot"])
waiting.head()

## Chart 3a: the waiting list over time

**Why this chart type?** A line, because the chart shows 92 monthly snapshots and the story is the movement of one number, the total, over time. Bars would be too crowded at this size. A stacked area would split the total into four bands, and only the bottom layer sits on a flat baseline, so the shape of the total would be harder to read. The y-axis starts at 450 thousand, not zero, because the list is large and fairly flat from zero, and this is said in the commentary under the chart. The line is broken at June 2021 because no snapshot exists for that month, and no value was estimated.

In [ ]:
total = (
    waiting.groupby("snapshot")["people"].sum() / 1000
)  # people on the list in each snapshot, in thousands

assert total.index.nunique() == 92  # 92 snapshots, January 2019 to September 2026
assert (round(total.iloc[0] * 1000), round(total.iloc[-1] * 1000)) == (523225, 692529)
total.head()

In [ ]:
before = total[total.index < "2021-06-01"]  # two lines, so the line is broken where June 2021 has no snapshot
after = total[total.index >= "2021-06-01"]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(before.index, before.values, color=ORANGE, linewidth=3)
ax.plot(after.index, after.values, color=ORANGE, linewidth=3)

ax.set_ylim(450, 700)
ax.set_yticks(range(450, 701, 25))  # a value every 25 thousand
ax.set_yticklabels([f"{v}k" for v in range(450, 701, 25)])
ax.set_xlim(total.index[0] - pd.Timedelta(days=60), total.index[-1] + pd.Timedelta(days=90))
ax.xaxis.set_major_locator(mdates.YearLocator())  # every year
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_ylabel("People on the list (thousands)")
ax.grid(axis="y", color="#efefef", linewidth=0.6)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

fig.text(
    0.01,
    0.95,
    "The outpatient waiting list, January 2019 to September 2026",
    fontsize=14,
    fontweight="bold",
    ha="left",
)
fig.text(0.01, 0.9, "It grew from 523,225 to 692,529 people, about 170,000 more.", fontsize=10.5, ha="left")
fig.text(0.01, 0.015, "Source: NTPF outpatient waiting list open data", fontsize=9, color=GREY, ha="left")
fig.subplots_adjust(top=0.82, bottom=0.1, left=0.08, right=0.97)
fig.savefig(FIGURES / "chart3a_waiting_list_over_time.png", dpi=200)
plt.show()

The national outpatient waiting list grew from 523,225 people in January 2019 to 692,529 in September 2026, an increase of about 170,000. It rose through 2020 and 2021, peaking at about 654,000 in September 2021. It then fell to about 562,000 by December 2023, and has risen again, from about 557,000 in December 2024 to its highest level in this data. There is no snapshot for June 2021, so the line is broken there rather than estimated.

Keep in mind: the y-axis starts at 450 thousand, not zero, so the movement of the list can be seen. The list counts people waiting at a date, not people treated, and it covers public hospital outpatients only.

## Chart 3b: how long people wait

**Why this chart type?** Grouped bars, because the question is how each waiting-time band changed across four years, and bars compare values side by side from a zero baseline. A stacked area chart would pile the four bands on top of each other, so only the bottom layer would be easy to compare, and the 18+ group, which is the story, would be hardest to read. The four greys run from light (2019) to dark (2026) so the years read in order, and blue is avoided because blue means cost in the other charts.

In [ ]:
bands = ["0-6 months", "6-12 months", "12-18 months", "18+ months"]
waiting["year"] = waiting["snapshot"].dt.year
monthly = waiting.groupby(["year", "snapshot", "band"])["people"].sum()
yearly = (
    monthly.groupby(["year", "band"]).mean().unstack()[bands] / 1000
)  # average thousands of people in each band
yearly = yearly.loc[[2019, 2021, 2024, 2026]]

assert [round(v) for v in yearly.loc[2021]] == [255, 119, 84, 178]
assert [round(v) for v in yearly.loc[2026]] == [391, 160, 65, 50]
yearly.round(1)

In [ ]:
greys = ["#D9D9D9", "#A6A6A6", "#737373", "#404040"]  # light (2019) to dark (2026)

fig, ax = plt.subplots(figsize=(10, 5.5))
yearly.T.plot(kind="bar", ax=ax, color=greys, width=0.8)
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=2, fontsize=9)  # the value on every bar

ax.set_ylim(0, 450)
ax.set_yticks(range(0, 451, 50))
ax.set_yticklabels([f"{v}k" for v in range(0, 451, 50)])
ax.set_xlabel("")
ax.set_ylabel("People waiting (thousands, yearly average)")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Year", frameon=False, ncol=4, loc="upper right")
ax.grid(axis="y", color="#efefef", linewidth=0.6)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

fig.text(0.01, 0.95, "Far fewer people now wait 18 months or more", fontsize=14, fontweight="bold", ha="left")
fig.text(
    0.01,
    0.9,
    "From about 178,000 people in 2021 to about 50,000 in 2026, while shorter waits grew.",
    fontsize=10.5,
    ha="left",
)
fig.text(
    0.01,
    0.015,
    "Source: NTPF outpatient waiting list open data. Yearly averages. 2026 covers January to September.",
    fontsize=9,
    color=GREY,
    ha="left",
)
fig.subplots_adjust(top=0.82, bottom=0.1, left=0.08, right=0.97)
fig.savefig(FIGURES / "chart3b_how_long_people_wait.png", dpi=200)
plt.show()

People waiting 18 months or more averaged about 178,000 in 2021, then fell to about 50,000 in 2024 and 2026, which is under 30% of the 2021 level. Shorter waits grew over the same period: the 0-6 month group went from about 262,000 in 2019 to about 391,000 in 2026, and the 6-12 month group reached its highest average in 2026. The 18+ group has stopped falling: 2025 was the low at about 45,000, and 2026 is slightly higher. 2026 only covers January to September.

Keep in mind: these are yearly averages of the monthly snapshots, in thousands. 2021 has no June snapshot, 2026 covers January to September only, and the tiny 'unknown' band is left out. 2025 is not shown but was the lowest year for waits of 18 months or more, at about 45,000.